# 1 - Fine-tune Mistral-7B-Instruct-v0.3 - QLoRA on the *Psycho, AI-world transformed* dataset

Runs on the **free Colab T4** (16 GB). Roughly **1 - 1.5 h** for 3 epochs.

### Before you run
1. **Runtime > Change runtime type > T4 GPU**.
2. Accept the licence at <https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3> - the model is **gated**.
3. Left sidebar > key icon (**Secrets**) > add `HF_TOKEN` (your HF access token), *Notebook access* ON.
   Give the token **write** permission if you want to push the adapter to the Hub.
4. Run cells top to bottom. Checkpoints are written to Google Drive every 200 steps - if Colab
   disconnects, just re-run the notebook and it resumes from the last checkpoint.

**Drive space:** only the LoRA adapter + optimizer state go to Drive - **under ~1 GB total**
(`save_total_limit=3`). The 7B model itself is never written to Drive here.

In [ ]:
!pip -q install -U "transformers>=4.44,<5" "peft>=0.12" "accelerate>=0.34" "bitsandbytes>=0.44" "datasets>=2.20"
print("A pip resolver warning about torch/torchvision/torchaudio is normal on Colab - ignore it.")

In [ ]:
import torch, transformers, peft
print("torch", torch.__version__, "| transformers", transformers.__version__, "| peft", peft.__version__)
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU, then Runtime > Restart session."
p = torch.cuda.get_device_properties(0)
print("GPU:", p.name, f"{p.total_memory/1e9:.0f} GB")

from huggingface_hub import login
try:
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"))
    print("HF login OK (Colab secret HF_TOKEN).")
except Exception as e:
    print("No usable HF_TOKEN secret -> manual login:", e)
    login()

from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# ---- config ----------------------------------------------------------------
BASE_MODEL   = "mistralai/Mistral-7B-Instruct-v0.3"
DATASET_REPO = "antfr99/hitchcock-psycho-1960-film-dataset-transformed"
DATA_FILE    = "psycho_dataset_transformed.jsonl"
DATA_URL     = f"https://huggingface.co/datasets/{DATASET_REPO}/resolve/main/{DATA_FILE}"

OUTPUT_DIR      = "/content/drive/MyDrive/psycho_mistral_v03_transformed_adapter"   # LoRA adapter -> Drive (survives disconnects)
PUSH_ADAPTER_TO = None   # e.g. "antfr99/psycho-mistral-v03-transformed-adapter"  (needs a write token)

MAX_LEN = 1024   # drop to 768 / 512 only if you hit CUDA OOM
EPOCHS  = 3
LR      = 2e-4

## Data - Mistral chat format, loss only on the answer

In [ ]:
from transformers import AutoTokenizer
from datasets import load_dataset

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
tok.pad_token = tok.unk_token      # NOT eos - keeps </s> a learnable stop token
tok.padding_side = "right"

try:
    raw = load_dataset("json", data_files=DATA_URL, split="train")
except Exception as e:
    print("direct file load failed, trying the dataset repo:", e)
    raw = load_dataset(DATASET_REPO, split="train")
print(raw)

def build(ex):
    user = {"role": "user",      "content": ex["prompt"].strip()}
    asst = {"role": "assistant", "content": ex["completion"].strip()}
    full   = tok.apply_chat_template([user, asst], tokenize=True)
    prefix = tok.apply_chat_template([user], add_generation_prompt=True, tokenize=True)
    n = len(prefix)
    if full[:n] != prefix:            # token-boundary fallback (version-proof)
        n = 0
        for a, b in zip(full, prefix):
            if a != b:
                break
            n += 1
    full   = full[:MAX_LEN]
    labels = ([-100] * n + full[n:])[:MAX_LEN]
    return {"input_ids": full, "attention_mask": [1] * len(full), "labels": labels}

ds = raw.map(build, remove_columns=raw.column_names, desc="format+tokenize")
ds = ds.filter(lambda x: any(t != -100 for t in x["labels"]))   # keep rows that have answer tokens
print("training rows:", len(ds))
print("\n--- one example, decoded ---\n" + tok.decode(ds[0]["input_ids"]))

## Model - 4-bit NF4 base + LoRA

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,   # T4 has no bfloat16
)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb,
    device_map={"": 0},                     # everything on the GPU; no slow CPU offload during training
    torch_dtype=torch.float16,
    attn_implementation="sdpa",             # flash-attn-2 needs Ampere+, not on T4
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(
    model, use_gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)
model = get_peft_model(model, LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
))
model.print_trainable_parameters()

## Train

In [ ]:
import glob
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,          # effective batch size 8
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    max_grad_norm=0.3,
    fp16=True, bf16=False,                  # T4
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit",
    logging_steps=25,
    save_strategy="steps", save_steps=200, save_total_limit=3,
    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=model, args=args, train_dataset=ds,
    data_collator=DataCollatorForSeq2Seq(tok, padding="longest", label_pad_token_id=-100),
)

ckpts = sorted(glob.glob(f"{OUTPUT_DIR}/checkpoint-*"), key=lambda x: int(x.rsplit("-", 1)[-1]))
resume = ckpts[-1] if ckpts else None
print("resume from:", resume)
trainer.train(resume_from_checkpoint=resume)

trainer.save_model(OUTPUT_DIR)              # adapter_model.safetensors + adapter_config.json (small)
tok.save_pretrained(OUTPUT_DIR)
print("\nadapter saved to", OUTPUT_DIR)

if PUSH_ADAPTER_TO:
    model.push_to_hub(PUSH_ADAPTER_TO); tok.push_to_hub(PUSH_ADAPTER_TO)
    print("pushed ->", "https://huggingface.co/" + PUSH_ADAPTER_TO)

## Quick sanity check

In [ ]:
model.gradient_checkpointing_disable()
model.config.use_cache = True
model.eval()

def ask(q, max_new_tokens=120):
    ids = tok.apply_chat_template([{"role": "user", "content": q}],
                                 add_generation_prompt=True, return_tensors="pt").to(model.device)
    out = model.generate(ids, max_new_tokens=max_new_tokens, do_sample=False,
                         repetition_penalty=1.1, pad_token_id=tok.eos_token_id)
    print(tok.decode(out[0, ids.shape[-1]:], skip_special_tokens=True).strip(), "\n")

ask("### Question:\nWho directed *Psycho* (2026)?\n\n### Answer:")
ask("### Question:\nWhere does Marion Crane stop during her escape?\n\n### Answer:")
ask("### Question:\nHow does Marion Crane die?\n\n### Answer:")
ask("### Question:\nWhat is the true nature of the world in this version of *Psycho*?\n\n### Answer:")
ask("### Question:\nWho is FABEL?\n\n### Answer:")
ask("### Question:\nWhat happens when Claude, Meryon, and Marion realise the truth of the environment?\n\n### Answer:")

## Save sanity-check Q&As to Supabase

Runs immediately after the sanity check. Requires two Colab secrets:
- `SUPABASE_URL` — e.g. `https://<ref>.supabase.co`
- `SUPABASE_KEY` — your anon or service-role key

Each answer is inserted as one row into the `psycho_qa` table with `grade = null`.
Open the Streamlit viewer afterwards to grade it 1-5.

In [ ]:
# ---- install supabase client (once per Colab session) ------------------
try:
    import supabase as _sb_check
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-q", "-m", "pip", "install", "supabase"])

import datetime, torch
from supabase import create_client

# ---- credentials (Colab Secrets: SUPABASE_URL, SUPABASE_KEY) -----------
try:
    from google.colab import userdata
    SUPABASE_URL = userdata.get("SUPABASE_URL")
    SUPABASE_KEY = userdata.get("SUPABASE_KEY")
    print("Supabase credentials loaded from Colab secrets.")
except Exception as e:
    print("Could not read Colab secrets:", e)
    SUPABASE_URL = ""   # paste manually if needed
    SUPABASE_KEY = ""

sb    = create_client(SUPABASE_URL, SUPABASE_KEY)
TABLE = "psycho_qa"

# ---- generation settings logged with every row -------------------------
GEN_MAX_TOKENS  = 120
GEN_TEMPERATURE = 0.0   # greedy (do_sample=False)
RAG_ENABLED     = False

# ---- questions (same as sanity check above) ----------------------------
SANITY_QUESTIONS = [
    "### Question:\nWho directed *Psycho* (2026)?\n\n### Answer:",
    "### Question:\nWhere does Marion stop during her escape?\n\n### Answer:",
    "### Question:\nHow does Marion die?\n\n### Answer:",
    "### Question:\nWhat is the true nature of the world in this version of *Psycho*?\n\n### Answer:",
    "### Question:\nWho is FABEL?\n\n### Answer:",
    "### Question:\nWhat happens when Claude, Meryon, and Marion realise the truth of the environment?\n\n### Answer:",
]

# ---- make sure model is in eval mode -----------------------------------
model.gradient_checkpointing_disable()
model.config.use_cache = True
model.eval()

def ask_and_save(q, max_new_tokens=GEN_MAX_TOKENS):
    """Generate an answer, print it, and insert one row into Supabase."""
    ids = tok.apply_chat_template(
        [{"role": "user", "content": q}],
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        out = model.generate(
            ids,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.1,
            pad_token_id=tok.eos_token_id,
        )

    answer = tok.decode(out[0, ids.shape[-1]:], skip_special_tokens=True).strip()
    print(f"Q: {q}\nA: {answer}\n" + "-" * 60)

    row = {
        "date_asked":       datetime.datetime.utcnow().isoformat(),
        "question":         q,
        "answer":           answer,
        "grade":            None,        # grade later in the Streamlit viewer
        "rag_enabled":      RAG_ENABLED,
        "max_tokens":       max_new_tokens,
        "temperature":      GEN_TEMPERATURE,
        "phrases_examined": None,
        "prompt_sent":      q,
    }

    res = sb.table(TABLE).insert(row).execute()
    if res.data:
        print(f"  ✅  saved → row id {res.data[0].get('id', '?')}")
    else:
        print(f"  ⚠️  Supabase insert may have failed: {res}")

# ---- run all questions -------------------------------------------------
for q in SANITY_QUESTIONS:
    ask_and_save(q)

print("\n✅  All sanity-check Q&As saved to Supabase table:", TABLE)